# Documentação — Notebook 05_ingestao_silver

Este notebook faz a **ingestão da camada Silver** para `ecommerce_clientes`. Lê o micro-lote novo da Bronze, aplica **10 regras de qualidade de dados**, separa registros válidos (Silver) de rejeitados (Quarentena) e calcula KPIs.

**Arquitetura dual (ADLS + UC fallback):** Tenta acessar o ADLS Gen2 primeiro (código original do Kauan). Se o ADLS estiver bloqueado (serverless), faz fallback automático para Unity Catalog + Lakehouse Federation.

## Célula 1 — Markdown de abertura

Descreve o notebook e lista as 10 regras de qualidade aplicadas:

| Regra | Tratamento |
| --- | --- |
| 1 | Colunas obrigatórias ausentes bloqueiam o lote; nulos vão para quarentena |
| 2 | E-mail único no lote e inexistente na Silver (quarentena) |
| 3 | `dt_cadastro` parseável e não futura (quarentena) |
| 4 | `uuid_cliente` em formato UUID v4 (quarentena) |
| 5 | Deduplicação de `id_cliente` e e-mail dentro do lote |
| 6 e 7 | KPIs: novos clientes em 10 min e cadastros por provedor |
| 8 | Alerta quando mesmo e-mail aparece em mais de um id_cliente |
| 9 | Tempo médio entre cadastro e primeira compra |
| 10 | `dt_ultima_atualizacao` não pode ser anterior a `dt_cadastro` |

## Célula 2 — Carregamento de credenciais

```python
%run ./00_setup_config
```

**O que faz:** Executa o notebook de setup, carregando as credenciais do `.env` (ADLS e SQL Server).

## Célula 3 — Importação de bibliotecas

```python
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import os
```

**O que faz:** Importa:
- `F` — funções Spark (filtros, agregações, hashes)
- `Window` — funções de janela (para deduplicação)
- `StructType/StructField` — definição de schema do log
- `datetime` — timestamps de processamento

## Célula 4 — Configuração de camadas (ADLS + UC fallback)

```python
# Opção 1: ADLS (original)
base_grupo = f"abfss://squad2@{storage_account_name}.dfs.core.windows.net/grupo2"
caminho_bronze = f"{base_grupo}/bronze/ecommerce_clientes"
...

# Opção 2: UC + Lakehouse Federation (fallback serverless)
tabela_bronze = "sqlserver_catalog.squad2.ecommerce_clientes"
tabela_silver = "workspace.default.silver_ecommerce_clientes"
...

# Teste de conectividade
try:
    spark.read.format("parquet").options(**adls_options).load(...).limit(1).count()
    usar_adls = True  # ADLS acessível
except Exception:
    usar_adls = False  # Fallback UC
```

**O que faz:** Define os caminhos de **ambas as opções** e faz um teste de conectividade no ADLS:
- Se ADLS acessível → `usar_adls = True` (usa caminhos ADLS)
- Se ADLS bloqueado → `usar_adls = False` (usa tabelas UC)

Todas as células subsequentes usam `if usar_adls: ... else: ...` para escolher a origem/destino.

## Célula 5 — Parâmetros de qualidade e função tabela_existe

```python
COLUNAS_OBRIGATORIAS = ["id_cliente", "uuid_cliente", "nome", "sobrenome", "email", "dt_cadastro"]
REGEX_UUID_V4 = r"^[0-9a-f]{8}-[0-9a-f]{4}-4[0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$"
PROVEDORES_CONHECIDOS = ["gmail", "yahoo", "hotmail", "outlook"]

def tabela_existe(caminho_ou_nome):
    if usar_adls:
        spark.read.format("delta").options(**adls_options).load(caminho_ou_nome).limit(1).count()
    else:
        spark.table(caminho_ou_nome).limit(1).count()
```

**O que faz:** Define os parâmetros das regras de qualidade e a função `tabela_existe` que funciona tanto para caminhos ADLS quanto para tabelas UC.

## Célula 6 — Leitura do micro-lote (incremental)

```python
if usar_adls:
    df_bronze = spark.read.format("delta").options(**adls_options).load(caminho_bronze)
    # Filtra por arquivos não processados (log de controle)
else:
    df_bronze = spark.table(tabela_bronze)
    # Filtra por id_cliente não presentes na Silver (anti-join)
```

**O que faz:** Lê a camada Bronze e filtra apenas os registros novos:
- **ADLS:** Controla por nome de arquivo no log de controle
- **UC fallback:** Controla por `id_cliente` com anti-join na Silver

Se não houver registros novos, o notebook encerra com `dbutils.notebook.exit()`.

## Células 7–9 — Regras de qualidade linha a linha

| Célula | Regra | O que faz |
| --- | --- | --- |
| 7 | Regra 1 | Verifica se todas as colunas obrigatórias existem. Se faltar alguma, BLOQUEIA o lote inteiro |
| 8 | Regras 1, 3, 4, 10 | Limpa textos (trim), valida e-mail/UUID, verifica dt_cadastro (não futura) e dt_atualizacao (não anterior a cadastro). Rejeitados vão para `df_rej_linha` |
| 9 | Regras 2, 5, 8 | Deduplica por id_cliente (mantém o mais recente) e detecta e-mails duplicados entre diferentes id_clientes |

**Resultado:** `df_ok` (válidos) e `df_rej_*` (rejeitados por motivo).

## Célula 10 — Comparação com a Silver existente

```python
if tabela_existe(...):
    df_silver_ref = spark.table(tabela_silver).select("id_cliente", "email").distinct()
    # Reenvios: id_cliente já na Silver → descarta
    # E-mail conflitante: e-mail na Silver com outro id_cliente → quarentena
```

**O que faz:** Compara o lote atual com a Silver já existente:
1. Se o `id_cliente` já está na Silver → descarta (reenvio)
2. Se o e-mail já existe na Silver com outro `id_cliente` → quarentena (`EMAIL_EXISTE_NA_SILVER`)

## Célula 11 — Montagem da Silver e Quarentena

```python
df_silver = df_ok.select(...).withColumn("silver_processed_at", F.current_timestamp())
df_quarentena = para_quarentena(df_rej_linha).unionByName(...)
```

**O que faz:** Monta os DataFrames finais:
- `df_silver` — registros válidos com timestamp de processamento
- `df_quarentena` — registros rejeitados com motivo de rejeição e timestamp

## Célula 12 — Resumo, alertas e KPI 7

**O que faz:** Calcula e imprime:
- Total de linhas lidas, duplicados, reenvios, quarentena e Silver
- Alertas das regras 8 e 10
- KPI 7: cadastros por provedor de e-mail (com alerta se >30% de provedores desconhecidos)

Tudo é calculado **antes** de gravar, porque os DataFrames dependem da Silver atual.

## Célula 13 — Gravação

```python
if usar_adls:
    df_quarentena.write.format("delta").options(**adls_options).mode("append").save(caminho_quarentena)
    df_silver.write.format("delta").options(**adls_options).mode("append").save(caminho_silver)
else:
    df_quarentena.write.format("delta").mode("append").saveAsTable(tabela_quarentena)
    df_silver.write.format("delta").mode("append").saveAsTable(tabela_silver)
```

**O que faz:** Grava os três artefatos (na ordem):
1. Quarentena (se houver rejeitados)
2. Silver (se houver válidos)
3. Log de controle (sempre, registra quais arquivos/origens foram processados)

A ordem importa: o log só é gravado se as gravações anteriores funcionarem. Se algo falhar, o lote é reprocessado na próxima execução.

## Célula 14 — KPI 6: Novos clientes em 10 minutos

```python
df_silver_atual = spark.table(tabela_silver)  # (ou ADLS)
novos_10min = df_silver_atual.filter(
    F.col("silver_processed_at") >= F.expr("current_timestamp() - INTERVAL 10 MINUTES")
).count()
```

**O que faz:** Conta quantos clientes foram processados na Silver nos últimos 10 minutos. Útil para monitorar a cadência do pipeline em tempo real.

## Célula 15 — KPI 9: Tempo até a primeira compra

```python
# ADLS: lê pedidos da RAW via Azure SDK (original)
# UC fallback: lê via Lakehouse Federation
if usar_adls:
    df_pedidos = ...  # Azure SDK + parquet
else:
    df_pedidos = spark.table("sqlserver_catalog.squad2.ecommerce_pedidos")
```

**O que faz:** Calcula o tempo médio entre o cadastro do cliente e sua primeira compra (pedidos não cancelados).
- Junta Silver (clientes) com pedidos
- Calcula `dias_ate_primeira_compra`
- Alerta se alguma compra ocorreu antes do cadastro

## Células 16–17 — Visualização e conferência

**Célula 16:** Mostra a Silver (schema + registros ordenados por data de processamento) e a quarentena (agrupada por motivo + registros).

**Célula 17:** Conferência final:
- Total de linhas vs. id_cliente únicos vs. e-mails únicos (devem ser iguais)
- Mostra datas da quarentena para confirmar que as futuras são realmente futuras

## Resumo do fluxo

| Célula | Função |
| --- | --- |
| 1 | Markdown com regras de qualidade |
| 2 | Carrega credenciais (%run 00_setup_config) |
| 3 | Importa bibliotecas |
| 4 | Configura camadas ADLS + UC + teste de conectividade |
| 5 | Parâmetros de qualidade + função tabela_existe |
| 6 | Lê micro-lote incremental da Bronze |
| 7 | Regra 1: valida schema |
| 8 | Regras 1, 3, 4, 10: limpeza e validação linha a linha |
| 9 | Regras 2, 5, 8: deduplicação e e-mail duplicado |
| 10 | Compara com Silver existente (reenvios + conflitos) |
| 11 | Monta Silver e quarentena |
| 12 | Resumo, alertas e KPI 7 |
| 13 | Grava quarentena, Silver e log |
| 14 | KPI 6: novos clientes em 10 min |
| 15 | KPI 9: tempo até primeira compra |
| 16 | Visualização Silver e quarentena |
| 17 | Conferência de integridade |

**Fluxo:** Configura → lê Bronze → valida → deduplica → compara → grava → KPIs → visualiza

**Tabelas UC criadas (fallback serverless):**
- `workspace.default.silver_ecommerce_clientes`
- `workspace.default.quarentena_ecommerce_clientes`
- `workspace.default.silver_control_log`